In [9]:
from pathlib import Path

from curriculum_learning.train import TrainingConfig, run_training

config = TrainingConfig(
    model_base_name="neuralmind/bert-base-portuguese-cased",
    train_batch_size=8,
    eval_batch_size=8,
    learning_rate=5e-5,
    num_train_epochs=10,
    seed=42,
    output_dir=Path("output"),
    name="exploration",
    group="default",
    curriculum_learning=False,
)

result = run_training(config, report_wandb=False)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Map:   0%|          | 0/6500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Roc Auc
1,0.462202,0.278775,0.892000,0.963456
2,0.300415,0.269989,0.914000,0.969600
3,0.283400,0.320626,0.922000,0.963984
4,0.168375,0.321451,0.926000,0.969008
5,0.130854,0.306153,0.946000,0.981792
6,0.099004,0.303614,0.948000,0.981416
7,0.064887,0.317933,0.942000,0.983376
8,0.048805,0.348537,0.948000,0.982176
9,0.038921,0.354912,0.948000,0.983640
10,0.018767,0.384190,0.944000,0.983392


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [4]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    PreTrainedTokenizerBase,
)

from curriculum_learning.analysis import compute_prediction_metrics, predict
from curriculum_learning.data import load_base_dataset, tokenize_dataset

# untrained base model, used as a baseline to compare against after fine-tuning
base_model = AutoModelForSequenceClassification.from_pretrained(
    config.model_base_name, num_labels=2
)
base_tokenizer = AutoTokenizer.from_pretrained(config.model_base_name)
if not isinstance(base_tokenizer, PreTrainedTokenizerBase):
    raise TypeError(
        f"Expected a PreTrainedTokenizerBase, got {type(base_tokenizer).__name__}"
    )

test_dataset = load_base_dataset()["test"]
tokenized_test_dataset = tokenize_dataset(test_dataset, base_tokenizer)

base_result = predict(base_model, base_tokenizer, tokenized_test_dataset)
print(compute_prediction_metrics(base_result))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'accuracy': 0.5, 'roc_auc': 0.44572867273270966}


In [5]:
import pandas as pd

pd.set_option("display.max_colwidth", None)

predictions_df = test_dataset.to_pandas().assign(
    true_label=base_result.labels,
    predicted_label=base_result.predictions,
    predicted_entailment_probability=base_result.probabilities,
)

predictions_df[
    [
        "sentence_pair_id",
        "premise",
        "hypothesis",
        "true_label",
        "predicted_label",
        "predicted_entailment_probability",
    ]
]

,sentence_pair_id,premise,hypothesis,true_label,predicted_label,predicted_entailment_probability
0,0,O cachorro caramelo está assistindo um cachorro castanho que está nadando em uma lagoa,"Um cachorro de estimação está de pé no banco e está olhando outro cachorro, que é castanho, na lagoa",0,0,0.333009
1,1,O cara está fazendo exercícios no chão,Um cara está fazendo exercícios,1,0,0.330351
2,2,Um cachorro grande e um cachorro pequenino estão parados ao lado do balcão da cozinha e estão investigando,Um cachorro grande e um cachorro pequenino estão de pé no balcão da cozinha e investigam,1,0,0.329612
3,3,Um menino jovem vestindo um traje de banho vermelho está pulando em uma piscina de crianças azul,Um menino jovem está vestindo um maiô vermelho e pulando para fora de uma piscina de criança,0,0,0.323420
4,4,Um cara velho com uma barba que é cinza está andando de bicicleta,Um cara velho com uma barba grisalha está andando de bicicleta,1,0,0.324865
...,...,...,...,...,...,...
2443,2443,Algum panda está escalando,Um urso está escalando,1,0,0.322435
2444,2444,Uma pessoa de topless com uma mochila está em frente a uma pilha de pedras e nuvens estão ao fundo,O homem está de pé em uma montanha rochosa e nuvens cinzas estão ao fundo,0,0,0.329083
2445,2445,Não tem nenhum homem cortando uma caixa,Um homem está cortando uma caixa,0,0,0.323869
2446,2446,Um cachorro preto está sentado na grama e mantendo a boca fechada,Um cachorro preto está correndo na grama,0,0,0.324315


In [11]:
from curriculum_learning.analysis import get_best_checkpoint

# fine-tuned model, loaded from the best checkpoint produced by training
checkpoint_dir = get_best_checkpoint(config.output_dir)
trained_model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
trained_tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
if not isinstance(trained_tokenizer, PreTrainedTokenizerBase):
    raise TypeError(
        f"Expected a PreTrainedTokenizerBase, got {type(trained_tokenizer).__name__}"
    )

tokenized_test_dataset_trained = tokenize_dataset(test_dataset, trained_tokenizer)

trained_result = predict(
    trained_model, trained_tokenizer, tokenized_test_dataset_trained
)
print(compute_prediction_metrics(trained_result))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/2448 [00:00<?, ? examples/s]

{'accuracy': 0.8500816993464052, 'roc_auc': 0.9195742022299117}


In [12]:
trained_predictions_df = test_dataset.to_pandas().assign(
    true_label=trained_result.labels,
    predicted_label=trained_result.predictions,
    predicted_entailment_probability=trained_result.probabilities,
)

trained_predictions_df[
    [
        "sentence_pair_id",
        "premise",
        "hypothesis",
        "true_label",
        "predicted_label",
        "predicted_entailment_probability",
    ]
]

,sentence_pair_id,premise,hypothesis,true_label,predicted_label,predicted_entailment_probability
0,0,O cachorro caramelo está assistindo um cachorro castanho que está nadando em uma lagoa,"Um cachorro de estimação está de pé no banco e está olhando outro cachorro, que é castanho, na lagoa",0,1,0.719224
1,1,O cara está fazendo exercícios no chão,Um cara está fazendo exercícios,1,1,0.977500
2,2,Um cachorro grande e um cachorro pequenino estão parados ao lado do balcão da cozinha e estão investigando,Um cachorro grande e um cachorro pequenino estão de pé no balcão da cozinha e investigam,1,1,0.946782
3,3,Um menino jovem vestindo um traje de banho vermelho está pulando em uma piscina de crianças azul,Um menino jovem está vestindo um maiô vermelho e pulando para fora de uma piscina de criança,0,1,0.891616
4,4,Um cara velho com uma barba que é cinza está andando de bicicleta,Um cara velho com uma barba grisalha está andando de bicicleta,1,1,0.907200
...,...,...,...,...,...,...
2443,2443,Algum panda está escalando,Um urso está escalando,1,0,0.043963
2444,2444,Uma pessoa de topless com uma mochila está em frente a uma pilha de pedras e nuvens estão ao fundo,O homem está de pé em uma montanha rochosa e nuvens cinzas estão ao fundo,0,1,0.710843
2445,2445,Não tem nenhum homem cortando uma caixa,Um homem está cortando uma caixa,0,0,0.007006
2446,2446,Um cachorro preto está sentado na grama e mantendo a boca fechada,Um cachorro preto está correndo na grama,0,1,0.937198
